# 1. Adquisición y preparación del conjunto de datos

Esta sección se encarga de la ingesta y preparación de los datos necesarios para el análisis de detección de fraude.

Se incluyen los pasos de carga de datos, limpieza, transformación y creación de variables relevantes para el modelo.

## Librerias necesarias

In [65]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import kagglehub

### Importando datos

In [ ]:
# Raíz del proyecto: primera carpeta (hacia arriba) que contiene pyproject.toml
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Crea data/raw/ si no existe (kagglehub exige una carpeta vacía, y data/ tiene .gitkeep)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Descarga la última versión en data/raw/ (sin usar la caché de kagglehub).
# Si ya se descargó, kagglehub lo detecta y no vuelve a descargar.
path = kagglehub.dataset_download(
    "mohankrishnathalla/credit-card-fraud-detection-dataset-2026",
    output_dir=str(RAW_DIR),
)

print("Path to dataset files:", path)
print("Archivos:", [f.name for f in RAW_DIR.iterdir() if not f.name.startswith(".")])

Path to dataset files: /Users/luishernandez/Desktop/MCD/2026-1/matematicas_mcd/fraud-detection-project/data/raw
Archivos: ['credit_card_fraud_detection_2026.csv']


## Datos iniciales

In [67]:
df = pd.read_csv(RAW_DIR / "credit_card_fraud_detection_2026.csv")
df.head()

,transaction_id,customer_id,card_id,merchant_id,transaction_datetime,transaction_amount,merchant_category,merchant_country,transaction_hour,transaction_day_of_week,...,browser,is_weekend,avg_transaction_amount_30d,std_transaction_amount_30d,num_declined_last_30d,num_chargebacks_ever,email_domain,is_fraud,fraud_type,fraud_amount_lost
0,TXN-00000001,CUS-00006042,CRD-00006042,MER-00002507,2026-01-01 02:28:51,37.36,Healthcare & Pharmacy,USA,2,Monday,...,Chrome,0,93.91,39.48,0,0,corporate,0,Legitimate,0.0
1,TXN-00000002,CUS-00005262,CRD-00005262,MER-00000189,2026-01-01 03:15:18,50.04,Restaurant & Dining,Germany,3,Monday,...,NaN,0,180.05,134.45,0,0,yahoo,0,Legitimate,0.0
2,TXN-00000003,CUS-00005901,CRD-00005901,MER-00002054,2026-01-01 03:44:38,38.87,Grocery & Supermarket,Australia,3,Monday,...,Chrome,0,42.00,22.46,0,0,hotmail,0,Legitimate,0.0
3,TXN-00000004,CUS-00006958,CRD-00006958,MER-00000310,2026-01-01 03:53:41,46.60,Gas Station,China,3,Monday,...,NaN,0,64.87,29.76,0,0,gmail,0,Legitimate,0.0
4,TXN-00000005,CUS-00003543,CRD-00003543,MER-00001197,2026-01-01 04:29:08,62.13,Clothing & Apparel,UK,4,Monday,...,NaN,0,47.19,18.52,0,0,gmail,0,Legitimate,0.0


### 1. Filtrar solo los fraudes de M´exico (columna merchant country).


In [68]:
df['merchant_country'].unique()

<StringArray>
[         'USA',      'Germany',    'Australia',        'China',
           'UK',        'Spain',       'France',       'Canada',
      'Nigeria',       'Brazil',       'Russia',       'Mexico',
        'Italy',  'Netherlands',        'Japan',          'UAE',
        'India',      'Romania',  'South Korea', 'South Africa']
Length: 20, dtype: str

In [69]:
df_mexico = df[df["merchant_country"] == "Mexico"]
df_mexico.head()

,transaction_id,customer_id,card_id,merchant_id,transaction_datetime,transaction_amount,merchant_category,merchant_country,transaction_hour,transaction_day_of_week,...,browser,is_weekend,avg_transaction_amount_30d,std_transaction_amount_30d,num_declined_last_30d,num_chargebacks_ever,email_domain,is_fraud,fraud_type,fraud_amount_lost
26,TXN-00000027,CUS-00001764,CRD-00001764,MER-00002391,2026-01-01 08:27:34,827.60,Hotel & Lodging,Mexico,8,Monday,...,NaN,0,90.02,65.89,0,0,gmail,0,Legitimate,0.0
54,TXN-00000055,CUS-00003611,CRD-00003611,MER-00002562,2026-01-01 09:56:35,62.19,Healthcare & Pharmacy,Mexico,9,Monday,...,NaN,0,100.98,46.77,0,0,yahoo,0,Legitimate,0.0
61,TXN-00000062,CUS-00007700,CRD-00007700,MER-00001370,2026-01-01 10:45:11,71.10,Hotel & Lodging,Mexico,10,Monday,...,App,0,79.40,58.88,0,0,yahoo,0,Legitimate,0.0
74,TXN-00000075,CUS-00002649,CRD-00002649,MER-00000119,2026-01-01 11:13:53,13.63,Restaurant & Dining,Mexico,11,Monday,...,Other,0,32.58,13.23,2,0,gmail,0,Legitimate,0.0
80,TXN-00000081,CUS-00007431,CRD-00007431,MER-00000790,2026-01-01 11:31:35,103.65,Electronics & Technology,Mexico,11,Monday,...,App,0,115.25,83.16,0,0,gmail,0,Legitimate,0.0


### Limpieza de datos

In [70]:
df_mexico.info() 

# La unica columna con valores nulos es browser que tiene 1085 valores no nulos de un total de 1921 registros.
#33  browser                      1085 non-null   str    
# sin embargo no la usaremos ya que solamente nos quedaremos con valores númericos

<class 'pandas.DataFrame'>
Index: 1921 entries, 26 to 99984
Data columns (total 43 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   transaction_id               1921 non-null   str    
 1   customer_id                  1921 non-null   str    
 2   card_id                      1921 non-null   str    
 3   merchant_id                  1921 non-null   str    
 4   transaction_datetime         1921 non-null   str    
 5   transaction_amount           1921 non-null   float64
 6   merchant_category            1921 non-null   str    
 7   merchant_country             1921 non-null   str    
 8   transaction_hour             1921 non-null   int64  
 9   transaction_day_of_week      1921 non-null   str    
 10  card_present                 1921 non-null   int64  
 11  customer_age                 1921 non-null   int64  
 12  customer_gender              1921 non-null   str    
 13  customer_state               192

### 2. Considerar solo las columnas numéricas y la columna is fraud.

In [71]:
num_cols = df_mexico.select_dtypes(include=["number"]).columns
num_df = df_mexico.select_dtypes(include=["number"])
if "is_fraud" not in num_df.columns:
    num_df = pd.concat([num_df, df_mexico["is_fraud"]], axis=1)


In [72]:
# Pasamos de 42 a 28 columnas con solo valores numéricos
num_df.head()

,transaction_amount,transaction_hour,card_present,customer_age,account_age_days,credit_limit,credit_score,num_cards_held,transactions_last_1h,transactions_last_24h,...,amount_vs_avg_ratio,used_chip,used_pin,is_weekend,avg_transaction_amount_30d,std_transaction_amount_30d,num_declined_last_30d,num_chargebacks_ever,is_fraud,fraud_amount_lost
26,827.60,8,1,26,188,15400.0,769,1,2,2,...,9.194,0,0,0,90.02,65.89,0,0,0,0.0
54,62.19,9,1,52,2129,14520.0,734,4,1,1,...,0.616,1,0,0,100.98,46.77,0,0,0,0.0
61,71.10,10,0,52,143,4730.0,561,2,1,5,...,0.895,0,0,0,79.40,58.88,0,0,0,0.0
74,13.63,11,0,46,387,4300.0,593,2,0,0,...,0.418,0,0,0,32.58,13.23,2,0,0,0.0
80,103.65,11,0,31,379,10650.0,671,2,0,3,...,0.899,0,0,0,115.25,83.16,0,0,0,0.0


In [73]:
num_df.info()

<class 'pandas.DataFrame'>
Index: 1921 entries, 26 to 99984
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   transaction_amount           1921 non-null   float64
 1   transaction_hour             1921 non-null   int64  
 2   card_present                 1921 non-null   int64  
 3   customer_age                 1921 non-null   int64  
 4   account_age_days             1921 non-null   int64  
 5   credit_limit                 1921 non-null   float64
 6   credit_score                 1921 non-null   int64  
 7   num_cards_held               1921 non-null   int64  
 8   transactions_last_1h         1921 non-null   int64  
 9   transactions_last_24h        1921 non-null   int64  
 10  transactions_last_7d         1921 non-null   int64  
 11  amount_last_1h               1921 non-null   float64
 12  amount_last_24h              1921 non-null   float64
 13  days_since_last_transaction  192

## Guardar el DataFrame preprocesado de México con solo columnas numéricas y la columna is_fraud

In [77]:
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

num_df.to_csv(PREPROCESSED_DIR / "mexico_fraud_num.csv", index=False)
